# Extract Airtable / Dropbox sources from Power BI semantic models

Loops through a known list of semantic models (datasets) in the Power BI / Fabric
Service, reads each model's **Power Query (M) code** via the XMLA / TOM endpoint,
and extracts every **Airtable** and **Dropbox** source definition so they can be
collated for a migration project.

**Inputs:** a CSV in this Lakehouse containing one row per model, with a
**workspace id** column and a **dataset id** column (GUIDs).

**Outputs (Delta tables in the attached Lakehouse):**
| table | contents |
|---|---|
| `migration_source_detail` | one row per (model, object, source_type) with matched values + full M code |
| `migration_source_summary` | distinct ultimate datasources and which models use them |
| `migration_scan_errors` | models that could not be scanned, with the error |

**Requirements**
* A **default Lakehouse** must be attached to this notebook (the CSV is read from it
  and the result tables are written to it).
* **XMLA endpoint = Read** enabled on the capacity, and the identity running the
  notebook must have at least *Read* on each semantic model (Build/Viewer).
* `semantic-link-labs` installed (next cell).

In [ ]:
# Install the TOM helper. Safe to re-run; restart not required within the session.
%pip install -q semantic-link-labs

## Parameters
This cell is tagged **parameters** so it can be overridden when the notebook is
run from a pipeline. Set `top_n` to a small number to test on a subset, or leave
it as `None` to process every model.

In [ ]:
# ---- CSV input ------------------------------------------------------------
csv_path        = "Files/migration/models.csv"  # path in the attached Lakehouse
workspace_id_col = "workspace_id"               # column holding the workspace GUID
dataset_id_col   = "dataset_id"                 # column holding the dataset  GUID

# ---- Testing knob ---------------------------------------------------------
top_n = 5            # int -> only process the first N models; None -> all of them

# ---- Output ---------------------------------------------------------------
output_prefix = "migration_"   # Delta tables get this prefix
write_tables  = True           # set False to only display, not persist

In [ ]:
import re
import datetime
import pandas as pd
import sempy.fabric as fabric
from sempy_labs.tom import connect_semantic_model

# --- patterns we care about ------------------------------------------------
AIRTABLE_URL   = re.compile(r'https?://(?:api\.)?airtable\.com/[^\s"\'\)\]\\]+', re.IGNORECASE)
AIRTABLE_BASE  = re.compile(r'\bapp[A-Za-z0-9]{14}\b')          # Airtable base id
AIRTABLE_TABLE = re.compile(r'\btbl[A-Za-z0-9]{14}\b')          # Airtable table id
DROPBOX_URL    = re.compile(r'https?://[^\s"\'\)\]\\]*dropbox[^\s"\'\)\]\\]*', re.IGNORECASE)
DROPBOX_QUOTED = re.compile(r'"([^"]*[Dd]ropbox[^"]*)"')        # quoted path/url containing Dropbox

def _extract(source_type, m_code):
    # Return a sorted list of distinct matched datasource values for one source type.
    vals = set()
    if source_type == "airtable":
        vals |= set(AIRTABLE_URL.findall(m_code))
        vals |= {"base:" + b for b in AIRTABLE_BASE.findall(m_code)}
        vals |= {"table:" + t for t in AIRTABLE_TABLE.findall(m_code)}
    else:  # dropbox
        vals |= set(DROPBOX_URL.findall(m_code))
        vals |= {v.strip() for v in DROPBOX_QUOTED.findall(m_code)}
    return sorted(vals)

def _classify(ws, ds, obj_type, obj_name, m_code):
    # Emit a detail row for each source type (airtable / dropbox) the M code touches.
    rows = []
    low = (m_code or "").lower()
    for stype in ("airtable", "dropbox"):
        if stype in low:
            rows.append({
                "workspace_id": ws,
                "dataset_id": ds,
                "object_type": obj_type,      # "Expression" or "Partition"
                "object_name": obj_name,
                "source_type": stype,
                "matched_values": " | ".join(_extract(stype, m_code)),
                "m_code": m_code,
            })
    return rows

def scan_model(ws, ds):
    # Connect to one semantic model and pull M from named expressions + partitions.
    rows = []
    with connect_semantic_model(dataset=ds, workspace=ws, readonly=True) as tom:
        # Shared / named expressions (parameters, reusable source functions, etc.)
        for e in tom.model.Expressions:
            rows += _classify(ws, ds, "Expression", e.Name, str(e.Expression))
        # Table partitions - only M partitions expose Source.Expression
        for t in tom.model.Tables:
            for p in t.Partitions:
                try:
                    expr = p.Source.Expression
                except Exception:
                    expr = None      # calculated / entity / DirectQuery partitions
                if expr:
                    rows += _classify(ws, ds, "Partition", t.Name + "." + p.Name, str(expr))
    return rows

In [ ]:
# Read the model list from the Lakehouse and apply the top_n test limit.
models_pdf = (
    spark.read.option("header", True).csv(csv_path)
    .select(workspace_id_col, dataset_id_col)
    .toPandas()
    .rename(columns={workspace_id_col: "workspace_id", dataset_id_col: "dataset_id"})
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

if top_n is not None:
    models_pdf = models_pdf.head(int(top_n))

print(f"{len(models_pdf)} model(s) to scan"
      + (f" (limited to top {top_n})" if top_n is not None else ""))
display(models_pdf)

In [ ]:
# Main loop: scan each model, collecting detail rows and any per-model errors.
detail_rows, error_rows = [], []
total = len(models_pdf)

for i, row in models_pdf.iterrows():
    ws, ds = row["workspace_id"], row["dataset_id"]
    try:
        found = scan_model(ws, ds)
        detail_rows.extend(found)
        n_at = sum(1 for r in found if r["source_type"] == "airtable")
        n_db = sum(1 for r in found if r["source_type"] == "dropbox")
        print(f"[{i + 1}/{total}] {ds}: {n_at} airtable, {n_db} dropbox object(s)")
    except Exception as ex:
        error_rows.append({"workspace_id": ws, "dataset_id": ds, "error": f"{type(ex).__name__}: {ex}"})
        print(f"[{i + 1}/{total}] {ds}: ERROR - {type(ex).__name__}: {ex}")

detail_pdf = pd.DataFrame(detail_rows, columns=[
    "workspace_id", "dataset_id", "object_type", "object_name",
    "source_type", "matched_values", "m_code"])
errors_pdf = pd.DataFrame(error_rows, columns=["workspace_id", "dataset_id", "error"])

print(f"\nDone. {len(detail_pdf)} source object(s) across {detail_pdf['dataset_id'].nunique() if len(detail_pdf) else 0} model(s); "
      f"{len(errors_pdf)} error(s).")
display(detail_pdf)

In [ ]:
# Build the distinct "ultimate datasources" summary by exploding matched_values.
summary_records = []
for _, r in detail_pdf.iterrows():
    for val in [v.strip() for v in str(r["matched_values"]).split("|") if v.strip()]:
        summary_records.append({"source_type": r["source_type"], "datasource": val,
                                "dataset_id": r["dataset_id"]})

if summary_records:
    sdf = pd.DataFrame(summary_records)
    summary_pdf = (
        sdf.groupby(["source_type", "datasource"])
        .agg(model_count=("dataset_id", "nunique"),
             dataset_ids=("dataset_id", lambda s: " | ".join(sorted(set(s)))))
        .reset_index()
        .sort_values(["source_type", "model_count"], ascending=[True, False])
    )
else:
    summary_pdf = pd.DataFrame(columns=["source_type", "datasource", "model_count", "dataset_ids"])

print(f"{len(summary_pdf)} distinct ultimate datasource(s).")
display(summary_pdf)

In [ ]:
# Persist results as Delta tables in the attached Lakehouse.
def _save(pdf, name):
    if pdf.empty:
        print(f"  (skipped {name}: no rows)")
        return
    (spark.createDataFrame(pdf)
        .write.mode("overwrite").option("overwriteSchema", "true")
        .format("delta").saveAsTable(name))
    print(f"  wrote {name} ({len(pdf)} rows)")

if write_tables:
    print("Writing tables:")
    _save(detail_pdf,  output_prefix + "source_detail")
    _save(summary_pdf, output_prefix + "source_summary")
    _save(errors_pdf,  output_prefix + "scan_errors")
else:
    print("write_tables=False - nothing persisted.")

## Notes & troubleshooting

* **`Connection cannot be established` / 401 / forbidden** - the notebook identity
  lacks access to that model, or XMLA Read is off. Grant Build/Viewer on the model
  (or add the running identity to the workspace) and enable *XMLA endpoint = Read*
  on the capacity. Such models land in `migration_scan_errors` rather than stopping
  the run.
* **Airtable via the generic Web connector** - if a model reaches Airtable through a
  parameterised base id rather than a literal `airtable.com` URL, the `app…`/`tbl…`
  id regexes still catch it from the M code, and the full `m_code` column lets you
  eyeball anything unusual.
* **Performance** - scanning is sequential (one XMLA session per model). 400 models
  typically runs in a handful of minutes; use `top_n` to validate on a few first.
* **Re-running** - tables are overwritten each run. Change `output_prefix` to keep
  separate test vs full outputs.